# VPython Colab transport probe

Run the cells **in order, one at a time**. Each JS probe prints its verdict
directly into its output. What we're testing:

1. Output-frame environment: does our JS run, can it load scripts from
   jsDelivr (how glow would ship — no extension, no nbextensions)?
2. **Comms path**: `google.colab.kernel.comms` round-trip kernel ↔ output
   (the ipywidgets mechanism — if this works, Colab is a comm-only
   frontend, the mirror image of VS Code's ws-only one).
3. **proxyPort path**: can an output frame open a WebSocket to a tornado
   server in the kernel via Colab's port proxy? (Folklore says no —
   let's get data.)

In [ ]:
# Cell 1: environment + jsDelivr + register the comm target (JS side)
from IPython.display import display, HTML
display(HTML('''
<div id="probe1" style="font-family:monospace;white-space:pre-wrap;border:1px solid #888;padding:6px"></div>
<script>
(function(){
  var out = document.getElementById('probe1');
  function log(m){ out.textContent += m + '\\n'; }
  log('probe v1 — JS is running in the output frame');
  var g = (typeof google !== 'undefined') ? google : null;
  log('google.colab:        ' + !!(g && g.colab));
  log('kernel.comms:        ' + !!(g && g.colab && g.colab.kernel && g.colab.kernel.comms));
  log('kernel.proxyPort:    ' + !!(g && g.colab && g.colab.kernel && typeof g.colab.kernel.proxyPort === 'function'));
  var s = document.createElement('script');
  s.src = 'https://cdn.jsdelivr.net/gh/vpython/vscode-vpython@main/media/jquery.min.js';
  s.onload = function(){ log('jsDelivr script load: OK (jQuery ' + (window.jQuery ? jQuery.fn.jquery : '?') + ')'); };
  s.onerror = function(){ log('jsDelivr script load: FAILED (CSP blocks external scripts?)'); };
  document.head.appendChild(s);
  if (g && g.colab && g.colab.kernel && g.colab.kernel.comms) {
    try {
      g.colab.kernel.comms.registerTarget('vpython-probe', function(comm, openMsg){
        log('comm OPEN from kernel; open data: ' + JSON.stringify((openMsg && openMsg.data) || openMsg || {}));
        log('comm object keys: ' + Object.keys(comm).join(', '));
        try { comm.send({echo: 'js-received-open', t: Date.now()}); log('comm.send back: called OK'); }
        catch(e){ log('comm.send threw: ' + e); }
        // API shape unknown in advance — try both message-receipt styles and
        // report which one this Colab build actually has.
        if (comm.on_message) {
          log('receive style: comm.on_message (callback)');
          comm.on_message(function(msg){ log('JS got message: ' + JSON.stringify(msg && msg.data || msg)); comm.send({echo2: 'roundtrip', got: (msg && msg.data) || null}); });
        } else if (comm.messages) {
          log('receive style: comm.messages (async iterator)');
          (async function(){
            try { for await (var msg of comm.messages) { log('JS got message: ' + JSON.stringify(msg && msg.data || msg)); comm.send({echo2: 'roundtrip', got: (msg && msg.data) || null}); } }
            catch(e){ log('messages iterator ended/threw: ' + e); }
          })();
        } else { log('NO known receive mechanism on comm object'); }
      });
      log('registerTarget(vpython-probe): OK — now run cell 2');
    } catch(e){ log('registerTarget threw: ' + e); }
  }
})();
</script>'''))

In [ ]:
# Cell 2: kernel side — open the comm (watch cell 1's box react)
from ipykernel.comm import Comm
probe_got = []
probe_comm = Comm(target_name='vpython-probe', data={'hello': 'from-kernel'})
probe_comm.on_msg(lambda m: probe_got.append(m['content']['data']))
print('comm opened; JS should have logged the OPEN. Now run cell 3.')

In [ ]:
# Cell 3: did the kernel receive the JS echo? Then send one downlink message.
print('kernel received so far:', probe_got)
probe_comm.send({'ping': 'kernel-to-js'})
print('sent ping — cell 1 box should log it; run this cell AGAIN to see echo2 arrive')

In [ ]:
# Cell 4: proxyPort probe, kernel side — tornado ws echo server
import socket, json, threading, asyncio
import tornado.ioloop, tornado.web, tornado.websocket, tornado.httpserver

def _free_port():
    s = socket.socket(); s.bind(('', 0))
    return s.getsockname()[1]

WS_PORT = _free_port()

class _Echo(tornado.websocket.WebSocketHandler):
    def on_message(self, message):
        self.write_message(json.dumps({'echo': json.loads(message), 'from': 'kernel-tornado'}))
    def check_origin(self, origin):
        return True

def _serve():
    asyncio.set_event_loop(asyncio.new_event_loop())
    app = tornado.web.Application([(r'/ws', _Echo)])
    tornado.httpserver.HTTPServer(app).listen(WS_PORT)
    tornado.ioloop.IOLoop.instance().start()

threading.Thread(target=_serve, daemon=True).start()
print('tornado ws echo on port', WS_PORT, '— now run cell 5')

In [ ]:
# Cell 5: proxyPort probe, JS side
from IPython.display import display, HTML
display(HTML('''
<div id="probe5" style="font-family:monospace;white-space:pre-wrap;border:1px solid #888;padding:6px"></div>
<script>
(async function(){
  var out = document.getElementById('probe5');
  function log(m){ out.textContent += m + '\\n'; }
  var PORT = ''' + str(WS_PORT) + ''';
  try {
    var base = await google.colab.kernel.proxyPort(PORT, {cache: false});
    log('proxyPort URL: ' + base);
    var wsurl = base.replace(/^http/, 'ws') + 'ws';
    log('trying WebSocket: ' + wsurl);
    var opened = false;
    var ws = new WebSocket(wsurl);
    ws.onopen = function(){ opened = true; log('WS OPEN ✅ — sending ping'); ws.send(JSON.stringify({hello:'via-proxy'})); };
    ws.onmessage = function(ev){ log('WS ECHO ✅: ' + ev.data); ws.close(); };
    ws.onerror = function(){ log('WS error event' + (opened ? ' (after open)' : ' — likely proxy rejects websockets')); };
    ws.onclose = function(ev){ log('WS closed, code ' + ev.code); };
    setTimeout(function(){ if (!opened) log('no WS connection after 8s ❌'); }, 8000);
    // control: does plain HTTP work through the same proxy? (isolates ws vs proxy)
    fetch(base + 'ws').then(function(r){ log('HTTP GET /ws through proxy: status ' + r.status + ' (any status = proxy itself works)'); })
                      .catch(function(e){ log('HTTP GET through proxy failed: ' + e); });
  } catch(e){ log('proxyPort threw: ' + e); }
})();
</script>'''))